In [1]:
%%writefile matrix_mul.cu
#include <iostream>
using namespace std;

// Kernel function
__global__ void multiply(int *A, int *B, int *C, int N) {
    int row = threadIdx.y;
    int col = threadIdx.x;

    if (row < N && col < N) {
        int sum = 0;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

int main() {
    int N = 2;

    int A[4] = {1, 2,
                3, 4};

    int B[4] = {5, 6,
                7, 8};

    int C[4];

    int *d_A, *d_B, *d_C;

    // Allocate GPU memory
    cudaMalloc(&d_A, N*N*sizeof(int));
    cudaMalloc(&d_B, N*N*sizeof(int));
    cudaMalloc(&d_C, N*N*sizeof(int));

    // Copy data to GPU
    cudaMemcpy(d_A, A, N*N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, N*N*sizeof(int), cudaMemcpyHostToDevice);

    // Launch kernel (1 block, NxN threads)
    dim3 threads(N, N);
    multiply<<<1, threads>>>(d_A, d_B, d_C, N);

    // Copy result back
    cudaMemcpy(C, d_C, N*N*sizeof(int), cudaMemcpyDeviceToHost);

    // Print Matrix A
    cout << "Matrix A:\n";
    for(int i=0;i<N;i++) {
        for(int j=0;j<N;j++) {
            cout << A[i*N + j] << " ";
        }
        cout << endl;
    }

    // Print Matrix B
    cout << "\nMatrix B:\n";
    for(int i=0;i<N;i++) {
        for(int j=0;j<N;j++) {
            cout << B[i*N + j] << " ";
        }
        cout << endl;
    }

    // Print Result
    cout << "\nMultiplication Result:\n";
    for(int i=0;i<N;i++) {
        for(int j=0;j<N;j++) {
            cout << C[i*N + j] << " ";
        }
        cout << endl;
    }

    // Free memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix_mul.cu


In [2]:
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!./matrix_mul

Matrix A:
1 2 
3 4 

Matrix B:
5 6 
7 8 

Multiplication Result:
19 22 
43 50 
